# SGEMM CUDA — Tesla T4 profiling (Google Colab)

Runs Simon Boehm's SGEMM kernels on a **Tesla T4** (Turing, `sm_75`) — the second GPU
for the cross-architecture comparison against the local **RTX 3050** (Ampere, `sm_86`).

### Before you run
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Upload **`SGEMM_CUDA.zip`** with the file browser (📁 in the left sidebar → *Upload*).
   This is the zip of the `SGEMM_CUDA/` folder described in `COLAB.md`.

Then **Runtime → Run all**, or run the cells top to bottom.

Only three things differ from the local RTX 3050 flow: compute capability is **75**,
there is **no `sudo`** (Colab runs as root), and tensor cores use **FP16 (kernel 15)**
instead of TF32/BF16 — Turing has no TF32/BF16 tensor-core path.

## 0 · Confirm the T4 is attached
Lists the active GPU — you should see **Tesla T4**. If not, set the runtime type to T4
(see above) and re-run this cell.

In [1]:
!nvidia-smi

Mon Sep  7 14:14:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1 · Unpack the code
Assumes `SGEMM_CUDA.zip` has already been uploaded to this session. Unzips it to
`/content/` and enters the project folder. `-o` overwrites on a re-run, so the cell is
safe to run again.

In [2]:
!unzip -q -o SGEMM_CUDA.zip -d /content
%cd /content/SGEMM_CUDA

/content/SGEMM_CUDA


## 2 · Check the toolchain
Colab already ships `nvcc` (the CUDA compiler) and `cmake`. **Nsight Compute** (`ncu`,
used for the E4 hardware-metric profiling) is often missing — the last line installs it
only if it isn't already present.

In [3]:
!nvcc --version | tail -2      # CUDA compiler (present by default)
!cmake --version | head -1     # build system (present by default)
!which ncu || apt-get -qq install -y nsight-compute   # profiler: install if missing

Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
cmake version 3.31.10
/usr/local/cuda/bin/ncu


## 3 · Build for the T4 (`sm_75`)
The target GPU architecture is a **cached** CMake variable, so it must be set on a
*fresh* build directory — hence `rm -rf build` first. `-DCUDA_COMPUTE_CAPABILITY=75`
targets the T4 (the default, 86, is for the RTX 3050). The last line is a cuBLAS smoke
test: it should print `GFLOPs` lines.

In [4]:
!rm -rf build
!cmake -B build -DCMAKE_BUILD_TYPE=Release -DCUDA_COMPUTE_CAPABILITY=75 .
!cmake --build build -j
!./build/sgemm 0

-- The CXX compiler identification is GNU 11.4.0
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
CMake Warning (dev) at CMakeLists.txt:5 (find_package):
  Policy CMP0146 is not set: The FindCUDA module is removed.  Run "cmake
  --help-policy CMP0146" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Th

## 4 · Runtime sweeps — E1 (square) · E2 (non-square) · E3 (arbitrary)
`run_all.sh` runs all three runtime experiments and writes per-kernel `.txt` files under
`results/t4/`. The kernel list drops **13/14** (TF32/BF16 — Ampere-only) and keeps **15**
(FP16), the tensor-core path that works on Turing.

Takes a few minutes. The T4's 16 GB also fits **N=8192**; to add that point, append
`8192` to `SIZES`, e.g. `SIZES="128,256,512,1024,2048,4096,8192"`.

In [5]:
!GPU=t4 KERNELS="0 1 2 3 4 5 6 7 8 9 10 15" bash scripts/run_all.sh

CMake Warning (dev) at CMakeLists.txt:5 (find_package):
  Policy CMP0146 is not set: The FindCUDA module is removed.  Run "cmake
  --help-policy CMP0146" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

== E1 square sweep ==
  [square] kernel 0
Running kernel 0 on device 0.
Max size: 4096
dimensions(m=128 n=128 k=128), alpha: 0.5, beta: 3
Average elapsed time: (0.000124) s, performance: (   33.8) GFLOPS. size: (128). M=128 N=128 K=128
dimensions(m=256 n=256 k=256), alpha: 0.5, beta: 3
Average elapsed time: (0.000255) s, performance: (  131.4) GFLOPS. size: (256). M=256 N=256 K=256
dimensions(m=512 n=512 k=512), alpha: 0.5, beta: 3
Average elapsed time: (0.000136) s, performance: ( 1970.5) GFLOPS. size: (512). M=512 N=512 K=512
dimensions(m=1024 n=1024 k=1024), alpha: 0.5, beta: 3
Average elapsed time: (0.000830) s, performance: ( 2588.3) GFLOPS. size: (1024). M=1024 N

## 5 · Hardware metrics — E4 (Nsight Compute)
Profiles kernels 1–6, 9 and 10 at N=4096 with `ncu --set full` (one launch each) and
dumps every metric to CSV. **No `sudo` and no driver unlock needed** — unlike the GeForce
RTX 3050, the T4 is a datacenter card and Colab runs as root. This is the slowest step
(each kernel is replayed ~40× to collect all metric groups).

The cell resolves `ncu`'s path and passes it to the script via `NCU=`.
_If you hit `ERR_NVGPUCTRPERM`, this Colab instance blocks counters — the runtime CSVs
above are the primary deliverable, and E4 can come from the RTX 3050 alone._

In [6]:
import os, shutil, glob
found = shutil.which("ncu") or next(iter(
    glob.glob("/opt/nvidia/nsight-compute/*/ncu") + glob.glob("/usr/local/cuda*/bin/ncu")), None)
os.environ["NCU"] = found or "ncu"
print("using ncu:", os.environ["NCU"])
!GPU=t4 bash scripts/profile.sh

using ncu: /usr/local/cuda/bin/ncu
== ncu kernel 1 (sgemm_naive) @ 4096 ==
==PROF== Connected to process 7580 (/content/SGEMM_CUDA/build/sgemm)
Running kernel 1 on device 0.
Max size: 4096
dimensions(m=4096 n=4096 k=4096), alpha: 0.5, beta: 3
==PROF== Profiling "sgemm_naive": 0%
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
....50%....100% - 31 passes
Average elapsed time: (2.222909) s, performance: (   61.8) GFLOPS. size: (4096). M=4096 N=4096 K=4096
==PROF== Disconnected from process 7580
==PROF== Report: /content/SGEMM_CUDA/results/t4/ncu/t4_k1_s4096.ncu-rep
== ncu kernel 2 (sgemm_global_mem_coalesce) @ 4096 ==
==PROF== Connected to process 9373 (/content/SGEMM_CUDA/build/sgemm)
Running kernel 2 on device 0.
Max size: 4096
dimensions

## 6 · Parameter-sensitivity sweep — E5
Rebuilds **kernel 9** for a grid of tile parameters (`BK`, `TM`, `TN`, …) to map the
performance cliff, then restores the original source. Writes
`results/t4/paramsweep/kernel9.csv`.

In [7]:
!GPU=t4 bash scripts/param_sweep.sh

9,128,128,8,2,2,LAUNCH_FAIL
9,128,128,8,4,4,469.6
9,128,128,8,8,8,3520.6
9,128,128,16,2,2,LAUNCH_FAIL
9,128,128,16,4,4,223.4
9,128,128,16,8,8,3497.8
9,128,128,32,2,2,LAUNCH_FAIL
9,128,128,32,4,4,2760.5
9,128,128,32,8,8,3400.5
9,128,128,64,2,2,BUILD_FAIL
9,128,128,64,4,4,BUILD_FAIL
9,128,128,64,8,8,BUILD_FAIL
DONE -> results/t4/paramsweep/kernel9.csv (src/runner.cu restored)


## 7 · Build combined CSVs (T4 sanity check)
Merges this session's raw dumps into two tidy tables so you can confirm they parse
cleanly on the T4. Both parsers are **standard-library only** (no pip installs).
`parse_ncu.py` also normalizes Nsight's auto-scaled units (time → ns, bytes → byte) so
each metric has one consistent unit.

These combined CSVs hold **T4 data only** — the final RTX 3050 + T4 merge happens locally
(step 9), which is why we download just the raw `results/t4/` folder.

In [8]:
!python3 analysis/collect_runtime.py   # -> results/combined/runtime_all.csv
!python3 analysis/parse_ncu.py         # -> results/combined/metrics_all.csv

wrote 404 rows -> results/combined/runtime_all.csv
wrote 272 rows -> results/combined/metrics_all.csv


## 8 · Quick preview (sanity check)
A glance at the two combined tables before downloading, to confirm they're populated.

In [9]:
import pandas as pd
rt = pd.read_csv("results/combined/runtime_all.csv")
mt = pd.read_csv("results/combined/metrics_all.csv")
print("runtime_all.csv:", rt.shape); display(rt.head())
print("metrics_all.csv:", mt.shape); display(mt.head())

runtime_all.csv: (404, 9)


,gpu,category,kernel,kernel_name,M,N,K,gflops,time_s
0,rtx3050,square,0,cuBLAS,128,128,128,4.7,0.000884
1,rtx3050,square,0,cuBLAS,256,256,256,1321.3,0.000025
2,rtx3050,square,0,cuBLAS,512,512,512,2995.2,0.000090
3,rtx3050,square,0,cuBLAS,1024,1024,1024,3747.6,0.000573
4,rtx3050,square,0,cuBLAS,2048,2048,2048,4233.3,0.004058


metrics_all.csv: (272, 7)


,gpu,kernel,size,metric,ncu_name,value,unit
0,rtx3050,10,4096,compute_sol_pct,sm__throughput.avg.pct_of_peak_sustained_elapsed,75.633615,%
1,rtx3050,10,4096,memory_sol_pct,gpu__compute_memory_throughput.avg.pct_of_peak...,60.930993,%
2,rtx3050,10,4096,dram_pct,gpu__dram_throughput.avg.pct_of_peak_sustained...,37.493745,%
3,rtx3050,10,4096,l2_pct,lts__throughput.avg.pct_of_peak_sustained_elapsed,24.667555,%
4,rtx3050,10,4096,l1_pct,l1tex__throughput.avg.pct_of_peak_sustained_el...,60.930993,%


## 9 · Download the T4 results
Zips **`results/t4/`** and downloads it. Unzip locally into `SGEMM_CUDA/results/` so
`rtx3050/` and `t4/` sit side by side, then re-run the two parser scripts **locally** —
they scan both GPUs and rebuild `results/combined/*.csv` with the full cross-GPU data
that `analysis/plots.py` expects.

In [10]:
!zip -qr t4_results.zip results/t4
from google.colab import files
files.download("t4_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>